# M3L2 E16 - RAG desde cero con LangChain

## Objetivo

En este notebook vas a construir un RAG completo desde cero y paso a paso.

RAG significa **Retrieval Augmented Generation**. En castellano: generacion aumentada con recuperacion.

La idea es simple:

1. Tenemos informacion propia en documentos.
2. Buscamos los fragmentos relevantes para una pregunta.
3. Le pasamos esos fragmentos al modelo.
4. El modelo responde usando ese contexto.

## Entregable

Al terminar deberias tener una chain RAG que:

- crea documentos de ejemplo,
- los divide en chunks,
- calcula embeddings,
- guarda los vectores en FAISS,
- recupera fragmentos relevantes,
- arma un prompt con contexto,
- llama al modelo,
- devuelve una respuesta final como string,
- permite inspeccionar cada parte del flujo.

## Mapa mental del RAG

```text
                 INGESTION

Texto crudo -> Document -> Splitter -> Chunks -> Embeddings -> Vector Store
                                                             |
                                                             v
                                                        Retriever

                  CONSULTA

Pregunta -> Retriever -> Docs relevantes -> Contexto -> Prompt -> LLM -> Parser -> Respuesta
```

Un error comun es pensar que RAG es "hacer una pregunta al modelo".

RAG no empieza en el modelo. RAG empieza antes: en como preparamos, partimos, indexamos y recuperamos documentos.

## Glosario minimo

| Termino | Que significa | En LangChain |
|---|---|---|
| Corpus | Conjunto total de textos disponibles | lista de documentos |
| Document | Texto + metadata | `Document(page_content=..., metadata=...)` |
| Metadata | Datos extra del documento | fuente, categoria, fecha |
| Chunk | Fragmento pequeno de un documento | salida del splitter |
| Splitter | Componente que divide texto | `RecursiveCharacterTextSplitter` |
| Embedding | Vector numerico que representa significado | `OpenAIEmbeddings` |
| Vector store | Base donde se guardan vectores | `FAISS` |
| Retriever | Interfaz para buscar docs relevantes | `as_retriever()` |
| k | Cantidad de docs a recuperar | `search_kwargs={"k": 3}` |
| Contexto | Texto recuperado que recibe el LLM | string con chunks |
| Prompt | Instruccion final al modelo | `ChatPromptTemplate` |
| Parser | Convierte salida del modelo | `StrOutputParser` |

## Por que RAG ayuda

Un LLM puede responder con conocimiento general, pero no conoce necesariamente tus documentos privados.

RAG reduce ese problema porque obliga al sistema a traer contexto antes de responder.

| Sin RAG | Con RAG |
|---|---|
| El modelo responde con memoria interna | El modelo recibe documentos relevantes |
| Puede inventar mas facil | Tiene contexto concreto |
| No se puede auditar de donde salio la respuesta | Podemos ver que chunks se recuperaron |
| Todo depende del prompt | El pipeline separa busqueda y generacion |

## Preparacion - Instalacion de dependencias

Esta celda instala las librerias necesarias si estas trabajando en Colab.

| Paquete | Para que se usa |
|---|---|
| `langchain` | Componentes base y LCEL |
| `langchain-openai` | ChatOpenAI y OpenAIEmbeddings |
| `langchain-community` | Integraciones como FAISS |
| `faiss-cpu` | Indice vectorial local en memoria |

La celda esta comentada para no reinstalar paquetes cada vez. En Colab se descomenta y se ejecuta una vez.

In [ ]:
# En Colab, ejecuta esta celda si faltan paquetes.
# !pip install langchain langchain-openai langchain-community faiss-cpu

## Preparacion - Cargar API key en Colab

Usamos `getpass` para pedir la API key sin dejarla escrita en el notebook.

La key se guarda en `os.environ["OPENAI_API_KEY"]` solo durante la sesion actual. Esto permite que `ChatOpenAI` y `OpenAIEmbeddings` funcionen sin usar `.env` ni `conexion.py`.

Este patron es mejor para Colab porque el notebook queda autocontenido y se puede compartir sin credenciales.

In [ ]:
import os
import getpass

if not os.getenv("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("Ingresa tu OpenAI API key: ")

print("API key cargada para esta sesion.")

## Paso 1 - Imports: que herramientas entran al RAG

Antes de escribir el pipeline, importamos las piezas que van a cumplir cada responsabilidad.

| Herramienta | Que es | Por que se usa en RAG |
|---|---|---|
| `Document` | Objeto de LangChain con texto y metadata | Representa cada fuente de conocimiento de forma ordenada |
| `RecursiveCharacterTextSplitter` | Splitter que corta texto en fragmentos | Evita mandar documentos largos completos y conserva contexto cercano |
| `OpenAIEmbeddings` | Modelo que convierte texto en vectores | Permite buscar por significado, no solo por palabras exactas |
| `FAISS` | Vector store en memoria | Guarda vectores y recupera textos similares rapido |
| `ChatPromptTemplate` | Template de prompt para chat | Ordena instrucciones, contexto y pregunta |
| `ChatOpenAI` | Wrapper del modelo de chat | Genera la respuesta final usando el contexto recuperado |
| `StrOutputParser` | Parser de salida | Convierte el mensaje del modelo en string simple |
| `RunnablePassthrough` | Componente LCEL que deja pasar el input | Permite usar la pregunta original dentro de la chain |

La idea profesional es que cada herramienta tenga una sola responsabilidad. Eso hace que el RAG sea mas facil de explicar, debuggear y cambiar.

In [ ]:
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

## Paso 2 - Crear el corpus: documentos y metadata

El corpus es el conjunto de textos sobre los que el sistema puede responder. En un producto real podria venir de PDFs, Notion, una base de datos, Google Drive o un CRM.

En este ejercicio usamos `Document` en memoria para que funcione directo en Colab sin subir archivos.

Cada `Document` tiene dos partes:

| Parte | Que guarda | Por que importa |
|---|---|---|
| `page_content` | El texto que se va a recuperar | Es el conocimiento que puede usar el modelo |
| `metadata` | Informacion extra como fuente o tema | Permite auditar de donde vino cada respuesta |

Usamos metadata porque un RAG no solo debe responder: tambien debe permitir revisar que fuentes uso.

In [ ]:
documentos = [
    Document(
        page_content="La politica de vacaciones indica que cada empleado tiene 15 dias habiles por ano calendario.",
        metadata={"fuente": "politicas_rrhh", "tema": "vacaciones"},
    ),
    Document(
        page_content="El trabajo remoto esta permitido hasta 3 dias por semana, coordinado previamente con el lider del equipo.",
        metadata={"fuente": "politicas_rrhh", "tema": "remoto"},
    ),
    Document(
        page_content="Los reintegros de capacitacion cubren hasta 300 dolares por trimestre para cursos relacionados con el rol.",
        metadata={"fuente": "beneficios", "tema": "capacitacion"},
    ),
    Document(
        page_content="La mesa de ayuda responde tickets criticos en menos de 4 horas habiles y tickets normales en 24 horas habiles.",
        metadata={"fuente": "soporte", "tema": "tickets"},
    ),
]

print("Documentos:", len(documentos))
for doc in documentos:
    print(doc.metadata, "->", doc.page_content[:80])

## Paso 3 - Dividir en chunks: por que no mandamos todo el documento

Un splitter divide documentos grandes en fragmentos mas pequenos llamados chunks.

Usamos `RecursiveCharacterTextSplitter` porque intenta cortar respetando separadores naturales del texto antes de cortar brutalmente por cantidad de caracteres.

Parametros importantes:

| Parametro | Que hace | Efecto practico |
|---|---|---|
| `chunk_size=180` | Tamano aproximado de cada chunk | Mantiene fragmentos chicos para este ejercicio |
| `chunk_overlap=30` | Repite una pequena parte entre chunks | Evita perder informacion cuando una idea queda entre dos cortes |

Si el chunk es muy grande, el retriever trae ruido. Si es muy chico, pierde contexto. Por eso este paso impacta directamente en la calidad del RAG.

In [ ]:
splitter = RecursiveCharacterTextSplitter(chunk_size=180, chunk_overlap=30)
chunks = splitter.split_documents(documentos)

print("Chunks generados:", len(chunks))
for i, chunk in enumerate(chunks, start=1):
    print(f"\n--- Chunk {i} ---")
    print("Metadata:", chunk.metadata)
    print(chunk.page_content)

## Paso 4 - Crear embeddings: convertir texto en significado numerico

Un embedding es un vector: una lista de numeros que representa el significado de un texto.

Usamos `OpenAIEmbeddings` para convertir preguntas y chunks al mismo espacio vectorial. Si dos textos significan algo parecido, sus vectores quedan cerca.

```text
"vacaciones" -> [0.012, -0.044, ...]
"dias libres" -> vector cercano
"soporte tecnico" -> vector mas lejano
```

Esto permite buscar por significado. El usuario no tiene que escribir exactamente las mismas palabras que aparecen en el documento.

In [ ]:
embeddings = OpenAIEmbeddings()

vector_demo = embeddings.embed_query("Cuantos dias de vacaciones tengo?")
print("Dimension del vector:", len(vector_demo))
print("Primeros 5 valores:", vector_demo[:5])

## Paso 5 - Crear el vector store: guardar chunks como vectores

El vector store es la estructura donde guardamos los chunks y sus embeddings.

Usamos `FAISS` porque es simple, rapido y funciona en memoria. Para clase y Colab es ideal porque no requiere configurar una base externa.

```text
chunks + embeddings -> indice FAISS
```

En produccion podriamos cambiar FAISS por Chroma, Pinecone, Weaviate, Supabase Vector u otro vector database. La idea del componente seria la misma.

In [ ]:
vectorstore = FAISS.from_documents(chunks, embeddings)
print(type(vectorstore))

## Paso 6 - Convertir FAISS en retriever: la interfaz de busqueda

El retriever es el componente que recibe una pregunta y devuelve documentos relevantes.

Usamos `as_retriever()` para no depender directamente de metodos especificos de FAISS. Asi el resto del pipeline puede hablar con una interfaz estandar:

```python
retriever.invoke(pregunta)
```

El parametro `k=3` significa: traer los 3 chunks mas relevantes.

| k bajo | k alto |
|---|---|
| Menos tokens y menos ruido | Mas contexto pero mas costo |
| Puede faltar informacion | Puede traer informacion irrelevante |

En un RAG real, ajustar `k` es una decision importante de calidad y costo.

In [ ]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

pregunta = "Cuantos dias de vacaciones tengo?"
docs_recuperados = retriever.invoke(pregunta)

for i, doc in enumerate(docs_recuperados, start=1):
    print(f"\nDoc recuperado {i}")
    print("Fuente:", doc.metadata.get("fuente"))
    print(doc.page_content)

## Paso 7 - Formatear contexto: pasar de Document a texto para el modelo

El retriever devuelve objetos `Document`, pero el modelo de chat no entiende objetos Python. El modelo recibe texto.

Por eso creamos `format_docs()`: toma los documentos recuperados y los convierte en un string de contexto.

Incluimos la fuente en el texto para que la respuesta pueda ser mas auditable:

```text
Fuente: politicas_rrhh
Contenido: La politica de vacaciones...
```

Este paso parece pequeno, pero es clave: define exactamente que ve el modelo antes de responder.

In [ ]:
def format_docs(docs):
    bloques = []
    for doc in docs:
        fuente = doc.metadata.get("fuente", "sin_fuente")
        bloques.append(f"Fuente: {fuente}\nContenido: {doc.page_content}")
    return "\n\n".join(bloques)


contexto = format_docs(docs_recuperados)
print(contexto)

## Paso 8 - Crear el prompt RAG: reglas para no inventar

El prompt RAG combina tres cosas:

1. Rol del asistente.
2. Regla de seguridad: responder solo con contexto.
3. Contexto recuperado + pregunta del usuario.

Usamos `ChatPromptTemplate` porque separa el texto fijo de las variables `{contexto}` y `{pregunta}`.

La instruccion mas importante es:

```text
Si la respuesta no aparece en el contexto, deci que no tenes informacion suficiente.
```

Esto no garantiza cero errores, pero reduce alucinaciones y hace el comportamiento mas claro para el alumno.

In [ ]:
prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "Sos un asistente de soporte interno. Responde solo con el contexto dado. "
        "Si la respuesta no aparece en el contexto, deci: No tengo informacion suficiente en los documentos.",
    ),
    (
        "human",
        "Contexto:\n{contexto}\n\nPregunta: {pregunta}",
    ),
])

print(prompt.input_variables)

## Paso 9 - Componer la chain RAG con LCEL: unir recuperacion y generacion

Ahora conectamos todos los componentes.

La chain tiene dos caminos en paralelo:

```text
pregunta -> retriever -> format_docs -> contexto
pregunta -----------------------------> pregunta original
```

Despues ambos valores entran al prompt:

```text
{contexto, pregunta} -> prompt -> llm -> parser -> respuesta
```

Usamos `RunnablePassthrough()` porque queremos conservar la pregunta original sin modificarla. Usamos `StrOutputParser()` porque queremos que la salida final sea un string y no un `AIMessage`.

Este es el punto donde RAG deja de ser pasos sueltos y se vuelve un pipeline reutilizable.

In [ ]:
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
parser = StrOutputParser()

rag_chain = (
    {
        "contexto": retriever | format_docs,
        "pregunta": RunnablePassthrough(),
    }
    | prompt
    | llm
    | parser
)

respuesta = rag_chain.invoke("Cuantos dias de vacaciones tengo?")
print(respuesta)

## Paso 10 - Debugging del RAG: como encontrar donde falla

Un RAG puede fallar por varias razones. No conviene mirar solo la respuesta final.

Debuggear RAG significa inspeccionar cada etapa:

| Si pasa esto | Revisar |
|---|---|
| La respuesta inventa | Prompt y contexto recuperado |
| No encuentra la respuesta | Retriever, embeddings, chunks y `k` |
| Trae documentos raros | Metadata, calidad del corpus y chunking |
| Responde incompleto | Chunk size, overlap y cantidad de docs recuperados |

En esta celda probamos varias preguntas y mostramos docs recuperados antes de mostrar la respuesta. Esa es la forma correcta de auditar un RAG.

In [ ]:
preguntas = [
    "Cuantos dias de vacaciones tengo?",
    "Cuantos dias de trabajo remoto puedo usar?",
    "La empresa tiene comedor gratis?",
]

for pregunta_actual in preguntas:
    print("\n==============================")
    print("Pregunta:", pregunta_actual)
    print("\nDocs recuperados:")
    for doc in retriever.invoke(pregunta_actual):
        print("-", doc.metadata.get("tema"), "|", doc.page_content)
    print("\nRespuesta:")
    print(rag_chain.invoke(pregunta_actual))

## Paso 11 - Funcion reutilizable: empaquetar el RAG como una unidad

Hasta ahora probamos el pipeline en celdas separadas. Para usarlo mejor, conviene envolverlo en una funcion.

`responder_con_rag()` devuelve un diccionario con:

| Campo | Para que sirve |
|---|---|
| `pregunta` | Auditar que se envio |
| `respuesta` | Mostrar al usuario final |
| `fuentes` | Saber que documentos participaron |
| `docs_recuperados` | Ver si el retriever trajo suficiente contexto |

Esta forma es mas parecida a lo que haria un backend: no devuelve solo texto, tambien devuelve datos utiles para trazabilidad.

In [ ]:
def responder_con_rag(pregunta: str) -> dict:
    docs = retriever.invoke(pregunta)
    respuesta = rag_chain.invoke(pregunta)
    fuentes = [doc.metadata.get("fuente", "sin_fuente") for doc in docs]
    return {
        "pregunta": pregunta,
        "respuesta": respuesta,
        "fuentes": fuentes,
        "docs_recuperados": len(docs),
    }


resultado = responder_con_rag("Que cubren los reintegros de capacitacion?")
print(resultado)

## Paso 12 - Checks finales: validar que el pipeline existe

Estos asserts no prueban la calidad semantica del RAG, pero si validan que todos los componentes principales fueron creados.

Chequeamos que existan documentos, chunks, embeddings, vector store, retriever, contexto, chain y respuesta.

Si alguno falla, significa que una etapa anterior quedo incompleta.

In [ ]:
assert len(documentos) >= 4
assert len(chunks) > 0
assert embeddings is not None
assert vectorstore is not None
assert retriever is not None
assert isinstance(contexto, str) and len(contexto) > 0
assert rag_chain is not None
assert isinstance(respuesta, str)
print("Checks OK")

## Resumen

Construiste un RAG completo:

```text
Document -> Splitter -> Embeddings -> FAISS -> Retriever -> Prompt -> LLM -> Parser
```

La ventaja es que cada parte se puede inspeccionar y reemplazar.